# 04 - Key Person(s) clause extraction via Claude Batch API

Adapted from `code/information_extraction/eds_forms/02_eds_extraction_claude.ipynb`.

**Differences from eds_forms:** input is the extracted **text** payload built by
`03_build_extraction_inputs.py` (not a rendered page image), and there is **one**
request per file (not the 3-query checkbox fan-out).

Reuses the batch registry, chunking, submit/retrieve, and `clean_json_response`
machinery near-verbatim. One JSON is written per file (resume with `clobber=False`).

Run modes come from `config.py` (`KP_MODE=test` for a cheap end-to-end dry run).

In [ ]:
import os, json, time
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple
import pandas as pd
import anthropic

import config as C

C.ensure_dirs()
api_key = C.read_api_key()
client = anthropic.Anthropic(api_key=api_key)

print(f'Mode:        {C.MODE}')
print(f'Model:       {C.MODEL}')
print(f'Clobber:     {C.CLOBBER}')
print(f'Inputs:      {C.INPUTS_DIR}')
print(f'Metadata:    {C.METADATA_CSV}')
print(f'Output JSON: {C.JSON_DIR}')

## System & user prompts

(The system prompt is ~600 tokens — below Haiku 4.5's 4096-token minimum cacheable
prefix, so prompt caching would be a no-op here. Cost control comes from the 50%
Batch API discount and sending only the relevant pages.)

In [ ]:
SYSTEM_PROMPT = """
You are an expert contract analyst. You are given the text of the page(s) of a
government contract that mention a "Key Person(s)" clause (the text may contain
OCR or PDF-extraction spacing artifacts such as "Person(s )" or "exc eed").

Find the Key Person(s) clause and classify it. The clause is often numbered and
titled, e.g. "29. Key Person(s).", and is frequently struck out with markers like
"- Deleted.", "(deleted)", "intentionally omitted", or "intentionally left blank".

Return ONLY valid JSON (no markdown, no commentary) in EXACTLY this shape:
{
  "key_persons_status": "present_named | present_unnamed | deleted | intentionally_omitted | not_found",
  "clause_section_number": "the leading section number as a string, e.g. 29, or null",
  "clause_text": "the key-person clause copied VERBATIM from the text",
  "named_individuals": ["full names of any specific key persons named, else empty array"],
  "confidence": "high | medium | low"
}

Status definitions:
- present_named: the clause is in force AND names specific individual(s).
- present_unnamed: the clause is in force but designates key persons by reference
  (e.g. "as listed in Section 33") without naming individuals here.
- deleted: the clause is struck out / marked deleted / removed.
- intentionally_omitted: marked "intentionally omitted / left blank / not applicable".
- not_found: no key-person language is actually present (false-positive match).

CRITICAL RULE FOR clause_text:
- Copy the clause text VERBATIM. Do NOT paraphrase, summarize, shorten, or reword it.
- You may ONLY normalize obvious extraction/OCR whitespace artifacts (collapse stray
  spaces, e.g. "Person(s )" -> "Person(s)") so the quote reads as it does in the
  contract. Preserve the original wording, punctuation, and section numbering.
- For deleted/omitted clauses, clause_text is the short struck-out line verbatim
  (e.g. "28. Key Person(s) - Deleted.").
- If status is not_found, use an empty string for clause_text.

Use an empty array [] (never null) for named_individuals when no one is named.
"""

USER_PROMPT = 'Classify the Key Person(s) clause in this contract text and return the JSON.'
print('Prompts loaded.')

## Batch registry (keyed by file_id) - copied from eds_forms, simplified

In [ ]:
def get_registry_path() -> Path:
    return C.JSON_DIR / 'batch_registry.json'

def load_batch_registry() -> Dict:
    p = get_registry_path()
    if not p.exists():
        return {'batches': {}, 'file_to_batch': {}}
    try:
        return json.loads(p.read_text())
    except Exception as e:
        print(f'Warning: could not load registry: {e}')
        return {'batches': {}, 'file_to_batch': {}}

def save_batch_registry(reg: Dict):
    get_registry_path().write_text(json.dumps(reg, indent=2))

def register_batch(batch_id, batch_file, timestamp, status, created_at, file_ids, num_requests):
    reg = load_batch_registry()
    reg['batches'][batch_id] = {
        'batch_file': batch_file, 'timestamp': timestamp, 'status': status,
        'created_at': created_at, 'num_requests': num_requests,
        'num_files': len(file_ids), 'retrieved': False, 'file_ids': list(file_ids),
    }
    for fid in file_ids:
        reg['file_to_batch'][fid] = batch_id
    save_batch_registry(reg)
    print(f'Registered batch {batch_id} with {len(file_ids)} files')

def mark_batch_retrieved(batch_id):
    reg = load_batch_registry()
    if batch_id in reg['batches']:
        reg['batches'][batch_id]['retrieved'] = True
        save_batch_registry(reg)

def get_pending_file_ids() -> set:
    # file_ids sitting in a not-yet-retrieved batch (load the registry ONCE;
    # callers must not re-read it per file).
    reg = load_batch_registry()
    pending = set()
    for info in reg['batches'].values():
        if not info.get('retrieved', False):
            pending.update(info.get('file_ids', []))
    return pending

def get_pending_batches() -> List[str]:
    reg = load_batch_registry()
    return [b for b, i in reg['batches'].items() if not i.get('retrieved', False)]

print('Registry functions loaded.')

## Request building, chunking, submit, retrieve

In [ ]:
def clean_json_response(text: str) -> str:
    if '```json' in text:
        s = text.find('```json') + 7; e = text.find('```', s); text = text[s:e].strip()
    elif '```' in text:
        s = text.find('```') + 3; e = text.find('```', s); text = text[s:e].strip()
    a, b = text.find('{'), text.rfind('}')
    if a != -1 and b != -1:
        text = text[a:b+1]
    return text.strip()

def create_batch_request(file_id: str, payload_text: str) -> Dict:
    # No cache_control: the system prompt is far below Haiku 4.5's 4096-token
    # minimum cacheable prefix, so a cache marker would silently do nothing.
    return {
        'custom_id': file_id,
        'params': {
            'model': C.MODEL,
            'max_tokens': C.MAX_TOKENS,
            'system': SYSTEM_PROMPT,
            'messages': [{'role': 'user', 'content': [
                {'type': 'text', 'text': payload_text},
                {'type': 'text', 'text': USER_PROMPT},
            ]}],
        },
    }

def estimate_request_size(req: Dict) -> int:
    return len(json.dumps(req).encode('utf-8'))

def chunk_requests(reqs: List[Dict], max_size_mb: int = 200) -> List[List[Dict]]:
    cap = max_size_mb * 1024 * 1024
    chunks, cur, size = [], [], 0
    for r in reqs:
        rs = estimate_request_size(r)
        if size + rs > cap and cur:
            chunks.append(cur); cur, size = [], 0
        cur.append(r); size += rs
    if cur:
        chunks.append(cur)
    return chunks

def submit_batch(reqs: List[Dict], file_ids: List[str], suffix: str = '') -> str:
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    batch_file = C.JSON_DIR / f'batch_{ts}{suffix}.jsonl'
    with open(batch_file, 'w') as f:
        for r in reqs:
            f.write(json.dumps(r) + '\n')
    batch = client.messages.batches.create(requests=reqs)
    print(f'Batch submitted! ID: {batch.id}  ({len(reqs)} requests)')
    register_batch(batch.id, batch_file.name, ts, batch.processing_status,
                   batch.created_at.isoformat() if batch.created_at else None,
                   file_ids, len(reqs))
    return batch.id

def submit_batches_chunked(all_reqs: List[Dict]) -> List[str]:
    chunks = chunk_requests(all_reqs)
    print(f'Splitting {len(all_reqs)} requests into {len(chunks)} batch(es)')
    ids = []
    for i, ch in enumerate(chunks, 1):
        suffix = f'_part{i}' if len(chunks) > 1 else ''
        ids.append(submit_batch(ch, [r['custom_id'] for r in ch], suffix))
    return ids

def check_batch_status(batch_id: str) -> Dict:
    b = client.messages.batches.retrieve(batch_id)
    return {'status': b.processing_status, 'counts': b.request_counts}

def retrieve_batch_results(batch_id: str) -> List:
    return list(client.messages.batches.results(batch_id))

print('Batch functions loaded.')

## Select files to process (clobber + pending-batch aware)

In [ ]:
def get_files_to_process() -> List[Tuple[str, str]]:
    # Return [(file_id, payload_text)] from the metadata CSV, skipping done/pending.
    if not C.METADATA_CSV.exists():
        print(f'ERROR: {C.METADATA_CSV} not found -- run 03_build_extraction_inputs.py first.')
        return []
    df = pd.read_csv(C.METADATA_CSV, dtype=str, keep_default_na=False)
    pending = get_pending_file_ids()
    # A curated test_ids.txt defines the whole test slice (it overrides
    # TEST_LIMIT, mirroring 01_scan_text_layer.py); the limit only caps the
    # fallback first-N test slice.
    limit = C.TEST_LIMIT if (C.MODE == 'test' and not C.load_test_ids()) else None
    out, skip_done, skip_pending, skip_missing = [], 0, 0, 0
    for _, row in df.iterrows():
        fid = row['file_id']
        out_json = C.JSON_DIR / f'{fid}.json'
        if out_json.exists() and not C.CLOBBER:
            skip_done += 1; continue
        if fid in pending:
            skip_pending += 1; continue
        payload = C.INPUTS_DIR / f'{fid}.txt'
        if not payload.exists():
            skip_missing += 1; continue
        out.append((fid, payload.read_text()))
        if limit and len(out) >= limit:
            print(f'[test] reached TEST_LIMIT ({limit})'); break
    print(f'To process: {len(out)} | skipped done={skip_done} pending={skip_pending} missing={skip_missing}')
    return out

print('Selection function loaded.')

## Process results -> one JSON per file

In [ ]:
def process_batch_results(results: List):
    saved = errored = 0
    for res in results:
        fid = res.custom_id
        if res.result.type != 'succeeded':
            print(f'Error in {fid}: {res.result.error}'); errored += 1; continue
        blocks = res.result.message.content
        if not blocks:
            print(f'Warning: empty response content for {fid}'); errored += 1; continue
        content = blocks[0].text
        try:
            parsed = json.loads(clean_json_response(content))
        except json.JSONDecodeError as e:
            # Saved with an 'error' key (no key_persons_status): step 06 reports
            # these as undetermined, never not_found.
            parsed = {'error': 'JSON parse failed', 'raw_response': content}
            print(f'Warning: JSON parse failed for {fid}: {e}')
        out = {
            'metadata': {'source_file': fid, 'extracted_at': datetime.now().isoformat(),
                         'model': C.MODEL},
            'extracted_data': parsed,
        }
        (C.JSON_DIR / f'{fid}.json').write_text(json.dumps(out, indent=2))
        saved += 1
    print(f'Saved {saved} JSON files ({errored} errored).')

print('Result processing loaded.')

## Run the pipeline (submit + poll + retrieve)

In [ ]:
def run_extraction_pipeline():
    files = get_files_to_process()
    if not files:
        print('No files to process!'); return []
    reqs = [create_batch_request(fid, text) for fid, text in files]
    batch_ids = submit_batches_chunked(reqs)
    print(f'\nBatch IDs: {batch_ids}')
    print('You can stop here and retrieve later with the cell below, or wait:')
    while True:
        done = True
        for bid in batch_ids:
            st = check_batch_status(bid)
            c = st['counts']
            print(f'  {bid}: {st["status"]}  succeeded={c.succeeded} '
                  f'processing={c.processing} errored={c.errored}')
            if st['status'] != 'ended':
                done = False
        if done:
            break
        time.sleep(300)
    for bid in batch_ids:
        process_batch_results(retrieve_batch_results(bid))
        mark_batch_retrieved(bid)
    print('\nExtraction complete.')
    return batch_ids

# batch_ids = run_extraction_pipeline()

## Retrieve later (if you stopped after submitting)

Run this cell any time after the batch(es) end to download and save results for
all pending batches without resubmitting.

In [ ]:
def retrieve_pending():
    pending = get_pending_batches()
    if not pending:
        print('No pending batches.'); return
    for bid in pending:
        st = check_batch_status(bid)
        if st['status'] != 'ended':
            print(f'  {bid}: {st["status"]} (not ready)'); continue
        process_batch_results(retrieve_batch_results(bid))
        mark_batch_retrieved(bid)
    print('Done retrieving pending batches.')

# retrieve_pending()